# Level 3 — Beyond the Paper: Your Research Project

### CAJAL Neuromics 2026 · Project 15: Mapping the Spatial Cellular Architecture of the Brain

**This is the open-ended level.** Levels 1–2 walked you along a fixed path: segment cells,
QC them, annotate the atlas, reproduce the niches and cell–cell communication of Wang et al.
(2025). Now the guardrails come off. You will **pick a question the paper did not fully
answer, design an analysis, run it, and present what you found** — even if the answer is
"it's inconclusive, and here is why."

**How this level works**

- **Kernel:** in JupyterLab, select the `Spatial Brain (SIF)` kernel (top-right), the same one used in Levels 1-2.
- **Time:** the largest block of the course (~2.5 days), ending in a short mini-presentation
  to the group (the course outline allots ~15 min per project — your instructor will confirm).
- **Deliverable:** a notebook that tells a story (question → approach → figures → interpretation
  → limitations) plus a few slides. Partial and negative results are expected and valued.
- **Scaffolding:** almost none. Below is a **menu of three projects**, then **one of them worked
  end-to-end as a reference** (Project 2). Use it as a template for rigour and structure — not as
  the thing to copy. Your instructor is your collaborator; ask early and often.

> **This is your working notebook.** Below the project menu is a blank workspace. Pick a project,
> fill in the planning template, and build your analysis from there. A fully worked reference for
> Project 2 lives in the *solution* notebook — try it yourself first; peek only when stuck.

---

### ⚠️ Where data lives — read before you load anything

Several people run this level **in parallel**, so be disciplined about paths:

- **READ** the large staged objects from the **shared, read-only** project folder
  (`/shared/projects/.../C15/data/...`). Never copy them into your repo, never write to that
  folder. They are staged once and shared by everyone.
- **WRITE** every output (figures, small tables, processed subsets) into **your own repo**
  (`figures/`, `data/`) via the `FilePaths` helper. Your outputs are yours alone.
- The multiome **ATAC** object is **16 GB** — do **not** `read_h5ad` it whole on a small
  allocation. Open it `backed="r"`, subset to the cells/peaks you need, *then* load. See
  Project 1's notes.

The staged files you can draw on:

| File | Size | Contents |
|---|---|---|
| `wang2025_merfish/processed/wang2025_merfish_cells.h5ad` | 0.19 GB | **404,030 annotated MERFISH cells** (the authors' cells, post-reveal). `X`=lognorm, `layers['counts']`; `obs`: `class/subclass/type/type_updated/niches/niche_name/Region/Sample_ID/Group/age`; `obsm`: `spatial`, `X_umap`. |
| `wang2025_multiome/processed/wang2025_multiome_rna.h5ad` | 3 GB | snRNA reference, 232,328 × 33,452, `X`=lognorm, `counts`, `X_pca`. |
| `wang2025_multiome/processed/wang2025_multiome_atac.h5ad` | 16 GB | snATAC, 232,328 × 82,505 peaks, `X`=TF-IDF, peak `chrom/start/end` in `var`. **Open backed.** |
| `wang2025_multiome/processed/wang2025_multiome.h5mu` | 9.9 GB | the paired MuData (RNA+ATAC together), if you need both modalities linked. |


## The project menu

Pick **one** (or propose your own — clear it with an instructor first). Each entry lists the
**question**, the **paper loose-end** it targets, which **data** to read, a **suggested approach**
with in-environment tools, a **compute note**, and what a **good deliverable** looks like.

All three are genuinely open — the paper touches each but leaves room. Difficulty and compute
weight differ; choose to match your interest and comfort with the tooling.

---

### Project 1 — The chromatin logic of a cell lineage *(snATAC / multiome)*

**Question.** L1–L2 only ever used gene *expression*. The atlas also has **snATAC — chromatin
accessibility over the same 232,328 nuclei** — that you have not touched. For a lineage you care
about (e.g. the excitatory-neuron trajectory RG → IPC-EN → EN-Newborn → EN-IT, or the Tri-IPC →
glia branch), **which regulatory regions and transcription factors define each step?**

**Loose-end.** The paper's regulatory story rests on SCENIC+ GRNs (582 eRegulons). You will not
re-run SCENIC+, but you can rebuild its logic from the raw ingredients and ask whether the master
regulators (e.g. *PAX6/EMX2* in radial glia, *TBR1/FOXP1* in EN, *LHX6/ARX* in IN) fall out.

**Data.** `wang2025_multiome_atac.h5ad` (peaks + genomic coords) **and**
`wang2025_multiome_rna.h5ad` (same cells, same order → paired).

**Suggested approach.**
- Open ATAC `backed="r"`; subset to your lineage's cells (and, if needed, to the most variable
  peaks) *before* loading into memory. ⚠️ This is slow off the shared filesystem — in our test just
  *opening* the 16 GB file backed took ~3 min, and pulling a two-cell-type subset (~14k cells) into
  memory took ~3 min more. Subsample cells generously and be patient.
- **Marker peaks:** `sc.tl.rank_genes_groups` on the peak matrix → differentially accessible
  regions per cell type.
- **Peak → gene links:** because RNA and ATAC are the *same* cells, correlate a peak's
  accessibility with a nearby gene's expression across cells (or across metacells you build by
  averaging within cell type × age). Peaks near a gene's TSS that track its expression are
  candidate enhancers. **You supply the gene's locus** (the RNA object has gene *symbols* but no
  coordinates — grab a TSS from a genome browser / GTF); the *peak* coords are already in
  `atac.var` (`chrom/start/end`).
- **TF activity:** `decoupler` with a TF–target network (e.g. CollecTRI) on the RNA gives a
  per-cell-type TF-activity matrix — a lightweight stand-in for eRegulon activity.
- *Stretch:* full GRN inference (SCENIC+/pycisTopic) — **not in the env**; you would add it and
  expect long CPU runtimes.

**Compute.** Heaviest project — memory-careful ATAC handling is the whole game. `decoupler`
network fetch needs internet (compute nodes have it). Subsample cells generously.

**Good deliverable.** A short list of cell-type-defining peaks/TFs for your lineage, at least one
convincing peak↔gene link with a plot, and an honest read on how well the accessibility story
matches the expression story.

---

### Project 2 — The neurogenesis-to-gliogenesis switch, in space *(MERFISH)* · ⭐ worked below

**Question.** The paper's headline is the **Tri-IPC** (tripotent intermediate progenitor,
ex-"IPC-glia") that, after ~GW18, locally produces **astrocytes, OPCs, and GABAergic
interneurons** — the moment the cortex pivots from making neurons to making glia. **Can you see
this transition in the tissue?** Where do Tri-IPCs and their putative progeny sit, and how does the
picture change across the three sampled ages (2nd trimester → 3rd trimester → infancy)?

**Loose-ends (two, verbatim).**
- *"Whether these late-born EN-Newborn cells will migrate to the cortical grey matter, the
  subcortical white matter or the olfactory bulb remains to be determined."*
- On IN-NCx_dGE-Immature (MEIS2⁺/SP8⁺): *"our spatial transcriptomic data indicate that [they]
  may become white-matter interneurons … further investigation is needed."*

**Data.** `wang2025_merfish_cells.h5ad` only — small, fast, low-memory, parallel-safe.

**Suggested approach.** Compositional shift across age; spatial maps of the lineage per age;
zone (`niche_name`) occupancy; `squidpy` neighbourhood enrichment / co-occurrence to test whether
Tri-IPCs physically sit next to their progeny. **No trajectory inference needed** — you use the
three age groups as a cross-sectional developmental axis. *(Fully worked below.)*

**Compute.** Light. Everything runs comfortably on a standard allocation (the neighbourhood
permutation test is the one slow-ish step, ~1–2 min).

**Good deliverable.** A figure panel showing the lineage's spatial/zonal distribution across age,
a neighbourhood-enrichment result, and a careful statement of what a fixed snapshot can and cannot
say about *fate*.

---

### Project 3 — A disease-risk map of the developing cortex *(multiome → space)*

**Question.** The paper maps neuropsychiatric **GWAS risk** onto cell types with SCAVENGE and finds,
strikingly, that **autism (ASD) risk peaks in 2nd-trimester immature intratelencephalic (IT)
neurons**, with a temporal ordering of disorders along differentiation. **Which developing cell
types — and which tissue locations — carry the most genetic risk for a disorder you choose?**

**Loose-end (verbatim).** On the cognition associations (*RG ↔ executive function, microglia ↔
working memory*): *"The exact mechanisms underlying these associations remain to be elucidated."*

**Data.** `wang2025_multiome_rna.h5ad` (per-type risk enrichment) + `wang2025_merfish_cells.h5ad`
(project the risk score into space). You supply risk gene sets: **SFARI Gene** (ASD) and the
**GWAS Catalog** are the standard sources.

**Suggested approach.**
- Assemble a risk gene set (start with a handful of high-confidence genes; scale up to the full
  SFARI/GWAS list). Keep it in *your* repo.
- Score it across cells with `decoupler` (or `sc.tl.score_genes`) → which cell types light up?
  Do you recover ASD → immature IT neurons? On the full-transcriptome **RNA reference** this is
  solid (all your risk genes are present).
- **Project into space:** score the same set on the MERFISH cells and plot the risk on tissue.
  ⚠️ **Panel limitation — check this first.** The MERFISH panel is only 300 genes: of a typical
  ~30-gene high-confidence ASD set, we found **only 4 on the panel** (DSCAM, CNTNAP2, RELN, SATB2).
  So the spatial projection rests on a *handful* of genes and is **illustrative at best** — quantify
  the overlap for your gene set up front, and if it's tiny, lean on the RNA-reference enrichment and
  present the spatial map as suggestive only.
- Compare two disorders, or a disorder vs a non-brain control trait, to show specificity.

**Compute.** Medium. RNA (3 GB) loads fine; scoring is cheap. This is an ATAC-free take on a
SCAVENGE-style idea (no accessibility propagation) — name that simplification honestly.

**Good deliverable.** A per-cell-type risk-enrichment plot, a (caveated) spatial risk map, and a
paragraph on what the *expression*-based proxy on a 300-gene panel can and cannot claim relative to
the paper's *accessibility*-based genome-wide method.

---

### Other directions (lighter, or bring-your-own data)

If none of the three grabs you, the paper and course leave other threads open — talk to an
instructor before committing:

- **Niche robustness** — the L2 niches came from one parameter choice (k-means on 50 neighbours,
  k=10). How stable are they to knn/radius/k? Which domains survive? *(MERFISH; light; `squidpy`.)*
- **A second reference** — annotate the MERFISH cohort with an *independent* developing-cortex atlas
  and compare concordance to the L2 labels. *(Needs an external reference staged.)*
- **Spatial denoising** — reference-based correction of transcript spillover (e.g. **SPLIT**,
  CPU-friendly), evaluated with the **L1 negative-marker purity metric** (import it from
  the `02_segmentation_and_metric` notebook rather than re-deriving).
- **Tri-IPC ↔ glioblastoma** — the paper shows >half of GBM malignant cells resemble Tri-IPCs. Map
  the Tri-IPC signature onto an external **glioblastoma** dataset for a disease angle. *(Needs a GBM
  reference downloaded to the shared folder — ask an instructor; not staged yet.)*

---

> The rest of this notebook works **Project 2** end to end, as a reference for the depth and
> structure we expect. Your project does not need to look like this one — but it should be this
> deliberate about **what it claims and what it cannot.**


# Your workspace

Start here. First, **fill in the planning template at the bottom** — a clear plan saves hours.
Then build your analysis in the cells below, adding markdown as you go to narrate what you find.

The cell below sets up the standard imports and the read-only data paths, so you don't have to
hunt for them. Uncomment the file(s) your project needs.

> **Reminder:** read the big objects from the shared read-only paths; write every output into
> **your** repo (`figures/`, `data/`). The ATAC file is 16 GB — open it `backed="r"` and subset
> before loading.

In [ ]:
import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401
import matplotlib.pyplot as plt  # noqa: F401
import seaborn as sns  # noqa: F401

import scanpy as sc
# import squidpy as sq          # spatial neighbourhood analysis (P2)
# import anndata as ad          # for backed reads (P1 ATAC)
# import decoupler as dc        # TF activity / gene-set scoring (P1, P3)

from spatialbrain import FilePaths

sc.settings.verbosity = 1
sc.set_figure_params(dpi=90, frameon=False, figsize=(5, 4))
%matplotlib inline

# --- Read-only staged data (shared; never write here) ---
DATA = "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C15/data"
MERFISH = f"{DATA}/wang2025_merfish/processed/wang2025_merfish_cells.h5ad"  # 0.19 GB, annotated (P2, P3)
RNA_REF = f"{DATA}/wang2025_multiome/processed/wang2025_multiome_rna.h5ad"  # 3 GB   (P1, P3)
ATAC = f"{DATA}/wang2025_multiome/processed/wang2025_multiome_atac.h5ad"  # 16 GB  -> open backed! (P1)

# --- Write outputs into YOUR repo ---
FIG = FilePaths.FIGURES / "level3"
FIG.mkdir(parents=True, exist_ok=True)
print("figures ->", FIG)

🔬 **Your analysis starts here.** Add cells as you need them.

In [ ]:
# Your analysis


---

# Your turn — planning template

Copy this into your own project section and fill it in **before** you write much code. A crisp plan
beats a big pile of exploratory cells.

**Project chosen:** _P1 / P2 / P3 / my own_

**Question (one sentence):** _…_

**Why it's open (the loose-end):** _…_

**Data I will read (exact files, read-only):** _…_

**Approach & tools (bullet the steps):**
- _…_

**What a positive result looks like — and a negative one:** _…_

**Biggest risk / thing most likely to go wrong:** _…_

---

## Presentation & write-up

Wrap up with a short mini-presentation to the group (the course outline allots ~15 min; your
instructor will confirm the format). A good arc:

1. **The question** and why the paper left it open (1 slide).
2. **Approach** — the data and the key method, briefly (1 slide).
3. **Results** — your two or three best figures, and what they show (2–3 slides).
4. **Interpretation & limits** — what you can and cannot claim (1 slide). *This is the most
   important slide.*
5. **What next** — the experiment or analysis that would actually settle it (1 slide).

Negative and partial results are genuinely valued: a clear "here's why the data can't answer this,
and here's what would" is a strong outcome.
